# Bot Forecast Log Generator - Tutorial

This notebook demonstrates how to generate a TSV log file from condensed forecast summaries.

## Overview

We'll extract metadata and forecast values from markdown files and create a structured dataset with:
- **Binary questions**: Single percentage values
- **Multiple Choice questions**: JSON dictionaries with option-percentage pairs
- **Numeric questions**: JSON dictionaries with percentile distributions

## Process Steps

1. Find all condensed summary files
2. Parse metadata from each file
3. Extract forecast values by question type
4. Combine into a pandas DataFrame
5. Export to TSV file

## Setup: Import Libraries

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path
from datetime import datetime

# Display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("✅ Libraries imported successfully")

## Step 1: Find Condensed Summary Files

We'll search the `all_forecast_summaries/` directory for files matching the pattern `*_condensed_*.md`.

In [ ]:
def find_condensed_summaries(source_dir='../all_forecast_summaries'):
    """Find all condensed summary markdown files"""
    source_path = Path(source_dir)
    files = sorted(source_path.glob('*_condensed_*.md'))
    return files

# Find files
files = find_condensed_summaries()

print(f"Found {len(files)} condensed summary files")
print("\nFirst 5 files:")
for f in files[:5]:
    print(f"  - {f.name}")

## Step 2: Parse Metadata

Each condensed summary has a metadata header with structured information:
- **Forecast ID**: Question identifier (we remove the 'q' prefix)
- **Question URL**: Full Metaculus URL
- **Question Type**: Binary, Multiple Choice, or Numeric
- **Forecast Date**: UTC timestamp (we split into date and time)

Let's look at an example file first:

In [ ]:
# Read and display a sample file (first 30 lines)
sample_file = files[0]
with open(sample_file, 'r', encoding='utf-8') as f:
    sample_content = f.read()

print(f"Sample file: {sample_file.name}")
print("=" * 80)
print('\n'.join(sample_content.split('\n')[:30]))
print("\n... (truncated)")

### Extract Metadata Function

Now let's create a function to extract metadata from the header section:

In [ ]:
def extract_metadata(content):
    """Extract metadata from FORECAST METADATA section"""
    metadata = {}
    
    # Find the metadata section
    metadata_section = re.search(
        r'# FORECAST METADATA\s*\n(.*?)\n---',
        content,
        re.DOTALL
    )
    
    if not metadata_section:
        return metadata
    
    section_text = metadata_section.group(1)
    
    # Extract all **Field**: value pairs
    pattern = r'\*\*(.+?)\*\*:\s*(.+?)(?:\n|$)'
    matches = re.findall(pattern, section_text)
    
    for field, value in matches:
        # Extract URL from markdown links [text](url)
        if field == 'Question URL':
            url_match = re.search(r'\[([^\]]*)\]\(([^\)]+)\)', value)
            if url_match:
                value = url_match.group(2)
        
        # Remove 'q' prefix from Forecast ID
        if field == 'Forecast ID' and value.startswith('q'):
            value = value[1:]
        
        metadata[field] = value.strip()
    
    return metadata

# Test on sample file
sample_metadata = extract_metadata(sample_content)

print("Extracted Metadata:")
for key, value in sample_metadata.items():
    print(f"  {key}: {value}")

### Split Date and Time

The Forecast Date field contains both date and time. We need to split them:

In [ ]:
def split_datetime(forecast_date):
    """Split forecast date into separate date and time fields"""
    # Parse formats like "2026-01-26 19:33:47 UTC"
    dt_match = re.match(r'(\d{4}-\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2})', forecast_date)
    if dt_match:
        return dt_match.group(1), dt_match.group(2)
    return None, None

# Test on sample
if 'Forecast Date' in sample_metadata:
    date, time = split_datetime(sample_metadata['Forecast Date'])
    print(f"Original: {sample_metadata['Forecast Date']}")
    print(f"Date: {date}")
    print(f"Time: {time}")

## Step 3: Extract Forecast Values

Forecast values have different formats depending on question type. Let's examine examples of each type:

### Find Examples of Each Question Type

In [ ]:
# Find one example of each question type
examples = {'Binary': None, 'Multiple Choice': None, 'Numeric': None}

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    metadata = extract_metadata(content)
    q_type = metadata.get('Question Type')
    
    if q_type in examples and examples[q_type] is None:
        examples[q_type] = (file, content, metadata)
    
    if all(v is not None for v in examples.values()):
        break

print("Examples found:")
for q_type, example in examples.items():
    if example:
        print(f"  {q_type}: {example[0].name}")

### Binary Question Example

Binary questions have a single percentage value.

In [ ]:
if examples['Binary']:
    file, content, metadata = examples['Binary']
    print(f"File: {file.name}")
    print(f"Question ID: {metadata.get('Forecast ID')}")
    print("\nForecast section:")
    
    # Find and display the forecast section
    forecast_section = re.search(
        r'# SUMMARY FORECAST VALUES\s*\n(.*?)\n##',
        content,
        re.DOTALL
    )
    if forecast_section:
        print(forecast_section.group(1)[:300])

In [ ]:
def extract_binary_forecast(content):
    """Extract forecast value for Binary questions"""
    pattern = r'\*Final Prediction\*:\s*(\d+\.?\d*)%?'
    match = re.search(pattern, content)
    if match:
        return match.group(1)
    return None

# Test on Binary example
if examples['Binary']:
    value = extract_binary_forecast(examples['Binary'][1])
    print(f"Extracted Binary value: {value}")

### Multiple Choice Question Example

Multiple Choice questions have a list of options with percentages. We convert this to a JSON dictionary, replacing spaces with underscores in option names.

In [ ]:
if examples['Multiple Choice']:
    file, content, metadata = examples['Multiple Choice']
    print(f"File: {file.name}")
    print(f"Question ID: {metadata.get('Forecast ID')}")
    print("\nForecast section:")
    
    forecast_section = re.search(
        r'# SUMMARY FORECAST VALUES\s*\n(.*?)\n##',
        content,
        re.DOTALL
    )
    if forecast_section:
        print(forecast_section.group(1)[:400])

In [ ]:
def extract_multiple_choice_forecast(content):
    """Extract forecast values for Multiple Choice questions"""
    # Find the Final Prediction section
    prediction_section = re.search(
        r'\*Final Prediction\*:\s*\n((?:- .+?:\s*\d+\.?\d*%?\s*\n?)+)',
        content,
        re.MULTILINE
    )
    
    if not prediction_section:
        return None
    
    section_text = prediction_section.group(1)
    
    # Extract option-percentage pairs
    pattern = r'-\s*(.+?):\s*(\d+\.?\d*)%?'
    matches = re.findall(pattern, section_text)
    
    if not matches:
        return None
    
    # Build dictionary with underscores replacing spaces
    forecast_dict = {}
    for option, percentage in matches:
        option = option.strip().replace(' ', '_')
        forecast_dict[option] = float(percentage)
    
    return json.dumps(forecast_dict)

# Test on Multiple Choice example
if examples['Multiple Choice']:
    value = extract_multiple_choice_forecast(examples['Multiple Choice'][1])
    print(f"Extracted Multiple Choice value:")
    print(value)
    print("\nParsed as dictionary:")
    print(json.loads(value))

### Numeric Question Example

Numeric questions have percentile distributions. We extract specific percentiles (p1, p5, p25, p50, p75, p95, p99) and store as a JSON dictionary.

In [ ]:
if examples['Numeric']:
    file, content, metadata = examples['Numeric']
    print(f"File: {file.name}")
    print(f"Question ID: {metadata.get('Forecast ID')}")
    print("\nForecast section:")
    
    forecast_section = re.search(
        r'# SUMMARY FORECAST VALUES\s*\n(.*?)\n##',
        content,
        re.DOTALL
    )
    if forecast_section:
        print(forecast_section.group(1)[:500])

In [ ]:
def extract_numeric_forecast(content):
    """Extract forecast values for Numeric questions"""
    # Pattern: XX.XX% chance of value below YY.YY
    pattern = r'(\d+\.?\d*)%\s+chance of value below\s+([\d.]+)'
    matches = re.findall(pattern, content)
    
    if not matches:
        return None
    
    # Map percentages to percentile keys
    percentile_map = {
        '1': 'p1', '1.0': 'p1', '1.00': 'p1',
        '5': 'p5', '5.0': 'p5', '5.00': 'p5',
        '25': 'p25', '25.0': 'p25', '25.00': 'p25',
        '50': 'p50', '50.0': 'p50', '50.00': 'p50',
        '75': 'p75', '75.0': 'p75', '75.00': 'p75',
        '95': 'p95', '95.0': 'p95', '95.00': 'p95',
        '99': 'p99', '99.0': 'p99', '99.00': 'p99'
    }
    
    # Build percentile dictionary
    forecast_dict = {}
    for percentile_str, value_str in matches:
        # Normalize percentile string
        percentile_normalized = percentile_str.rstrip('0').rstrip('.')
        
        if percentile_normalized in percentile_map:
            key = percentile_map[percentile_normalized]
            forecast_dict[key] = float(value_str)
    
    if not forecast_dict:
        return None
    
    return json.dumps(forecast_dict)

# Test on Numeric example
if examples['Numeric']:
    value = extract_numeric_forecast(examples['Numeric'][1])
    print(f"Extracted Numeric value:")
    print(value)
    print("\nParsed as dictionary:")
    print(json.loads(value))

## Step 4: Process All Files

Now let's combine everything into a single function that processes each file and returns a row of data:

In [ ]:
def process_file(file_path):
    """Process a single condensed summary file and extract all data"""
    # Read file
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Extract forecast number from filename
    forecast_number = re.match(r'^(\d+)_', file_path.name)
    if not forecast_number:
        return None
    forecast_number = forecast_number.group(1)
    
    # Extract metadata
    metadata = extract_metadata(content)
    
    # Get required fields
    question_type = metadata.get('Question Type')
    question_url = metadata.get('Question URL')
    forecast_date = metadata.get('Forecast Date')
    
    if not all([question_type, question_url, forecast_date]):
        return None
    
    # Split date and time
    date, time = split_datetime(forecast_date)
    if not date or not time:
        return None
    
    # Extract forecast values based on question type
    if question_type == 'Binary':
        forecast_values = extract_binary_forecast(content)
    elif question_type == 'Multiple Choice':
        forecast_values = extract_multiple_choice_forecast(content)
    elif question_type == 'Numeric':
        forecast_values = extract_numeric_forecast(content)
    else:
        forecast_values = None
    
    # Build row dictionary
    row = {
        'run': '',  # Empty, to be populated externally
        'forecast_number': forecast_number,
        'date': date,
        'time': time,
        'question_type': question_type,
        'tournament_name': tournament_name,
        'question_url': question_url,
        'comments': '',  # Empty, for user annotations
        'forecast': forecast_values,
        'community_forecast': '',  # Empty, to be populated externally
        'score': '',  # Empty, to be populated externally
        'community_score': ''  # Empty, to be populated externally
    }
    
    return row

# Test on a few files
print("Testing on first 3 files:")
for file in files[:3]:
    row = process_file(file)
    if row:
        print(f"\n{file.name}:")
        print(f"  Type: {row['question_type']}")
        print(f"  Date: {row['date']} {row['time']}")
        print(f"  Values: {row['forecast'][:80]}..." if len(str(row['forecast'])) > 80 else f"  Values: {row['forecast']}")

### Process All Files and Create DataFrame

In [ ]:
# Process all files
rows = []
errors = []

for file in files:
    row = process_file(file)
    if row:
        rows.append(row)
    else:
        errors.append(file.name)

# Create DataFrame
df = pd.DataFrame(rows)

# Sort by date and time (oldest to newest)
df = df.sort_values(['date', 'time'])
df = df.reset_index(drop=True)

print(f"✅ Processed {len(rows)} files successfully")
if errors:
    print(f"⚠️  {len(errors)} files had errors: {errors}")

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## Step 5: Explore the Data

Let's examine the resulting DataFrame:

### View First Few Rows

In [ ]:
# Display first 5 rows
df.head()

### Summary Statistics by Question Type

In [ ]:
# Count by question type
type_counts = df['question_type'].value_counts()
print("Forecast counts by question type:")
print(type_counts)

# Create a summary DataFrame
summary_df = pd.DataFrame({
    'Question Type': type_counts.index,
    'Count': type_counts.values,
    'Percentage': (type_counts.values / len(df) * 100).round(1)
})

print("\nSummary:")
summary_df

### View Examples of Each Question Type

In [ ]:
# Binary examples
print("Binary Question Examples:")
binary_df = df[df['question_type'] == 'Binary'][['forecast_number', 'date', 'time', 'forecast']].head(3)
display(binary_df)

In [ ]:
# Multiple Choice examples
print("Multiple Choice Question Examples:")
mc_df = df[df['question_type'] == 'Multiple Choice'][['forecast_number', 'date', 'forecast']].head(3)
display(mc_df)

# Show parsed JSON for one example
if len(mc_df) > 0:
    print("\nParsed JSON for first Multiple Choice forecast:")
    first_mc = mc_df.iloc[0]['forecast']
    print(json.dumps(json.loads(first_mc), indent=2))

In [ ]:
# Numeric examples
print("Numeric Question Examples:")
numeric_df = df[df['question_type'] == 'Numeric'][['forecast_number', 'date', 'forecast']].head(3)
display(numeric_df)

# Show parsed JSON for one example
if len(numeric_df) > 0:
    print("\nParsed JSON for first Numeric forecast:")
    first_numeric = numeric_df.iloc[0]['forecast']
    print(json.dumps(json.loads(first_numeric), indent=2))

### Check for Empty Fields

Verify that `run` and `comments` columns are empty as expected:

In [ ]:
# Check empty fields
print("Empty field verification:")
print(f"  run column all empty: {df['run'].eq('').all()}")
print(f"  comments column all empty: {df['comments'].eq('').all()}")

# Check for any missing forecast values
missing_values = df['forecast'].isna().sum()
print(f"  Missing forecast values: {missing_values}")

## Step 6: Export to TSV

Finally, let's save the DataFrame as a TSV file:

In [ ]:
# Define output path
output_path = Path('../all_forecast_summaries/bot_forecast_log.tsv')

# Export to TSV
df.to_csv(output_path, sep='\t', index=False)

print(f"✅ Exported to: {output_path.absolute()}")
print(f"   Rows: {len(df)}")
print(f"   Size: {output_path.stat().st_size:,} bytes")

### Verify TSV Output

Let's read the TSV back in to verify it was saved correctly:

In [ ]:
# Read TSV back in
df_verify = pd.read_csv(output_path, sep='\t')

print(f"TSV verification:")
print(f"  Rows: {len(df_verify)}")
print(f"  Columns: {list(df_verify.columns)}")
print(f"\nFirst 3 rows:")
display(df_verify.head(3))

## Bonus: Data Analysis Examples

Now that we have the data in a DataFrame, let's do some quick analysis:

### Average Binary Forecast Value

In [ ]:
# Convert Binary forecast values to float
binary_forecasts = df[df['question_type'] == 'Binary'].copy()
binary_forecasts['value_numeric'] = binary_forecasts['forecast'].astype(float)

print("Binary Forecast Statistics:")
print(f"  Count: {len(binary_forecasts)}")
print(f"  Mean: {binary_forecasts['value_numeric'].mean():.2f}%")
print(f"  Median: {binary_forecasts['value_numeric'].median():.2f}%")
print(f"  Min: {binary_forecasts['value_numeric'].min():.2f}%")
print(f"  Max: {binary_forecasts['value_numeric'].max():.2f}%")

# Display distribution
binary_forecasts[['forecast_number', 'date', 'value_numeric']].sort_values('value_numeric')

### Multiple Choice: Extract Most Common Option

In [ ]:
# Parse Multiple Choice forecasts and find highest probability option
mc_forecasts = df[df['question_type'] == 'Multiple Choice'].copy()

def get_top_option(forecast_json):
    """Get the option with highest probability"""
    data = json.loads(forecast_json)
    top_option = max(data.items(), key=lambda x: x[1])
    return top_option[0], top_option[1]

mc_forecasts[['top_option', 'top_probability']] = mc_forecasts['forecast'].apply(
    lambda x: pd.Series(get_top_option(x))
)

print("Multiple Choice: Top Options")
display(mc_forecasts[['forecast_number', 'date', 'top_option', 'top_probability']])

### Numeric: Extract Median Values

In [ ]:
# Parse Numeric forecasts and extract p50 (median)
numeric_forecasts = df[df['question_type'] == 'Numeric'].copy()

def get_median(forecast_json):
    """Extract p50 (median) from numeric forecast"""
    data = json.loads(forecast_json)
    return data.get('p50', None)

numeric_forecasts['median'] = numeric_forecasts['forecast'].apply(get_median)

print("Numeric: Median Values (p50)")
display(numeric_forecasts[['forecast_number', 'date', 'median']])

## Summary

We successfully:
1. ✅ Found all condensed summary files
2. ✅ Extracted metadata (forecast ID, date, time, question type, URL)
3. ✅ Parsed forecast values for all three question types
4. ✅ Created a structured DataFrame
5. ✅ Exported to TSV format
6. ✅ Performed basic data analysis

### Key Takeaways

- **Binary questions**: Single percentage value (no % sign)
- **Multiple Choice questions**: JSON dictionary with option-percentage pairs
  - Spaces in option names are replaced with underscores
- **Numeric questions**: JSON dictionary with percentile distributions
  - Common percentiles: p1, p5, p25, p50, p75, p95, p99

### Next Steps

- Populate the `run` column by merging with GitHub Actions metadata
- Add custom annotations to the `comments` column
- Perform deeper analysis on forecast patterns
- Visualize forecast distributions over time